# PINN Calorimeter Notebook

Geant4 calorimeter surrogate comparison:
1. CNN supervised surrogate
2. Physics-informed CNN surrogate
3. Conditional GAN

Input HDF5 branches:
    /default_ntuples/hits/layer/pages
    /default_ntuples/hits/edep/pages
    /default_ntuples/hits/x/pages
    /default_ntuples/hits/y/pages
    /default_ntuples/hits/event/pages

Edit FILES below.

Run the cells from top to bottom. Edit `FILES` in the configuration cell if your HDF5 files live elsewhere.

## Imports and Configuration

Loads the scientific Python, plotting, HDF5, and PyTorch dependencies used throughout the notebook. The `%matplotlib inline` magic makes plots render directly in Jupyter output cells.

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from scipy.optimize import curve_fit
from scipy.special import gamma as gamma_func

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split


## Config

Defines the input HDF5 files, calorimeter grid geometry, training device, batch size, epoch counts, optimizer learning rate, and the physics-loss weights. Edit `FILES` here if your datasets are in a different folder.

In [ ]:
# ============================================================
# CONFIG
# ============================================================

FILES = {
    5:   "electron_5GeV.h5",
    10:  "electron_10GeV.h5",
    20:  "electron_20GeV.h5",
    30:  "electron_30GeV.h5",
    50:  "electron_50GeV.h5",
    75:  "electron_75GeV.h5",
    100: "electron_100GeV.h5",
}

N_LAYERS = 50
NX = 32
NY = 32

XMIN, XMAX = -100.0, 100.0  # mm
YMIN, YMAX = -100.0, 100.0  # mm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
EPOCHS_CNN = 20
EPOCHS_PINN = 20
EPOCHS_GAN = 30

LR = 1e-4

LAMBDA_E = 0.1
LAMBDA_LONGO = 0.01
LAMBDA_RADIAL = 0.01


## Hdf5 Loading + Voxelization

Provides utilities for reading Geant4 hit branches from HDF5 files and accumulating hit energy into fixed `[event, layer, x, y]` voxel grids. It also defines the `Dataset` wrapper used by PyTorch data loaders.

In [ ]:
# ============================================================
# HDF5 LOADING + VOXELIZATION
# ============================================================

def load_hits(path):
    with h5py.File(path, "r") as f:
        layer = np.array(f["/default_ntuples/hits/layer/pages"]).astype(np.int64)
        edep  = np.array(f["/default_ntuples/hits/edep/pages"]).astype(np.float32)
        x     = np.array(f["/default_ntuples/hits/x/pages"]).astype(np.float32)
        y     = np.array(f["/default_ntuples/hits/y/pages"]).astype(np.float32)

        event_path = "/default_ntuples/hits/event/pages"
        if event_path not in f:
            raise RuntimeError(
                "No event branch found. Add event ID to the HDF5 file or change event_path."
            )
        event = np.array(f[event_path]).astype(np.int64)

    return event, layer, edep, x, y


def voxelize(event, layer, edep, x, y):
    n_events = int(event.max()) + 1
    grid = np.zeros((n_events, N_LAYERS, NX, NY), dtype=np.float32)

    ix = ((x - XMIN) / (XMAX - XMIN) * NX).astype(np.int64)
    iy = ((y - YMIN) / (YMAX - YMIN) * NY).astype(np.int64)

    mask = (
        (layer >= 0) & (layer < N_LAYERS) &
        (ix >= 0) & (ix < NX) &
        (iy >= 0) & (iy < NY) &
        (edep > 0)
    )

    np.add.at(
        grid,
        (event[mask], layer[mask], ix[mask], iy[mask]),
        edep[mask],
    )

    return grid


def build_arrays(files):
    X_logE = []
    E0_all = []
    Y_all = []

    for E0, path in files.items():
        print(f"Loading {E0} GeV from {path}")
        event, layer, edep, x, y = load_hits(path)
        vox = voxelize(event, layer, edep, x, y)

        evis = vox.sum(axis=(1, 2, 3))
        mask = evis > 0
        vox = vox[mask]

        n = len(vox)

        # Since 50 layers are assumed contained, normalize by incident E0.
        vox_norm = vox / float(E0)

        X_logE.append(np.full((n, 1), np.log(E0), dtype=np.float32))
        E0_all.append(np.full((n,), float(E0), dtype=np.float32))
        Y_all.append(vox_norm.astype(np.float32))

    X_logE = np.concatenate(X_logE)
    E0_all = np.concatenate(E0_all)
    Y_all = np.concatenate(Y_all)

    return X_logE, E0_all, Y_all


class CaloDataset(Dataset):
    def __init__(self, X_logE, E0, Y):
        self.X = torch.tensor(X_logE, dtype=torch.float32)
        self.E0 = torch.tensor(E0, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.E0[idx], self.Y[idx]


## Physics Profiles

This section turns known calorimeter-shower behavior into numerical targets that can be used during training. A plain supervised CNN can learn voxel-by-voxel averages from the data, but it has no explicit reason to preserve global shower physics such as where the shower peaks along the detector depth, how energy spreads laterally, or whether the predicted shower shape remains plausible for a given incident beam energy.

The Longo/gamma fit summarizes the average longitudinal development of electromagnetic showers for each input energy. Instead of forcing the network to memorize one event at a time, this gives the PINN loss a smooth reference profile for how deposited energy should be distributed across layers. The radial template plays a similar role in the transverse direction: it provides a compact expectation for how shower energy should fall off with distance from the shower axis.

These profiles are not replacing the Geant4 training labels. They add weak physics constraints on top of the voxel reconstruction loss, helping the model produce showers that are statistically consistent with calorimeter physics even when individual voxel predictions are noisy, sparse, or underconstrained.

In [ ]:
# ============================================================
# PHYSICS PROFILES
# ============================================================

def fit_longo_targets(Y, E0, dt=0.386):
    """
    Fit one Longo/gamma profile per energy from voxelized Geant4 data.
    Y is normalized by E0.
    """

    def longo(t, a, b, norm):
        return norm * b * (b * t) ** (a - 1) * np.exp(-b * t) / gamma_func(a)

    targets = {}
    energies = sorted(np.unique(E0))

    for energy in energies:
        mask = E0 == energy
        Y_e = Y[mask]

        long = Y_e.sum(axis=(2, 3))
        long = long / (long.sum(axis=1, keepdims=True) + 1e-12)
        mean_long = long.mean(axis=0)

        t = (np.arange(N_LAYERS) + 0.5) * dt

        popt, _ = curve_fit(
            longo,
            t,
            mean_long,
            p0=[4.0, 0.6, 1.0],
            bounds=([0.5, 0.05, 0.0], [30.0, 10.0, 10.0]),
            maxfev=20000,
        )

        a, b, norm = popt
        prof = longo(t, a, b, norm)
        prof = prof / prof.sum()

        targets[float(energy)] = torch.tensor(prof, dtype=torch.float32)

        tmax = (a - 1.0) / b
        print(
            f"Longo fit E={energy:.0f} GeV: "
            f"a={a:.3f}, b={b:.3f}, tmax={tmax:.2f} X0"
        )

    return targets


def radial_template(n_bins=24, Rm=0.20):
    r = torch.linspace(0.0, 1.0, n_bins)
    p = r * torch.exp(-r / Rm)
    p = p / (p.sum() + 1e-12)
    return p


def radial_project(Y, n_bins=24):
    """
    Y: [B, L, NX, NY]
    returns [B, n_bins]
    """

    B, L, nx, ny = Y.shape
    xs = torch.linspace(-1.0, 1.0, nx, device=Y.device)
    ys = torch.linspace(-1.0, 1.0, ny, device=Y.device)
    xx, yy = torch.meshgrid(xs, ys, indexing="ij")
    rr = torch.sqrt(xx**2 + yy**2)

    out = []
    for i in range(n_bins):
        rmin = i / n_bins
        rmax = (i + 1) / n_bins
        mask = ((rr >= rmin) & (rr < rmax)).float()
        val = (Y * mask[None, None, :, :]).sum(dim=(1, 2, 3))
        out.append(val)

    return torch.stack(out, dim=1)


def get_longo_batch(E0_batch, longo_targets):
    profiles = []
    for e in E0_batch.detach().cpu().numpy():
        key = float(int(round(e)))
        profiles.append(longo_targets[key])
    return torch.stack(profiles).to(E0_batch.device)


## Models

Defines the neural network architectures: a conditional CNN generator for deterministic shower prediction, a noise-conditioned GAN generator, and a discriminator that scores generated versus Geant4 showers.

In [ ]:
# ============================================================
# MODELS
# ============================================================

class CNNGenerator(nn.Module):
    """
    Conditional CNN decoder:
    log(E0) -> [50, 32, 32]
    """

    def __init__(self, latent_dim=256):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(1, 128),
            nn.SiLU(),
            nn.Linear(128, latent_dim),
            nn.SiLU(),
            nn.Linear(latent_dim, 128 * 4 * 4),
            nn.SiLU(),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 4, 2, 1),  # 8x8
            nn.SiLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 16x16
            nn.SiLU(),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),    # 32x32
            nn.SiLU(),
            nn.Conv2d(64, N_LAYERS, kernel_size=3, padding=1),
            nn.Softplus(),
        )

    def forward(self, logE):
        z = self.fc(logE)
        z = z.view(-1, 128, 4, 4)
        return self.decoder(z)


class GANGenerator(nn.Module):
    def __init__(self, noise_dim=64):
        super().__init__()
        self.noise_dim = noise_dim

        self.fc = nn.Sequential(
            nn.Linear(noise_dim + 1, 256),
            nn.SiLU(),
            nn.Linear(256, 128 * 4 * 4),
            nn.SiLU(),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 4, 2, 1),
            nn.SiLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.SiLU(),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),
            nn.SiLU(),
            nn.Conv2d(64, N_LAYERS, kernel_size=3, padding=1),
            nn.Softplus(),
        )

    def forward(self, logE, z):
        h = torch.cat([logE, z], dim=1)
        h = self.fc(h)
        h = h.view(-1, 128, 4, 4)
        return self.decoder(h)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(N_LAYERS, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.LeakyReLU(0.2),
        )

        self.fc = nn.Sequential(
            nn.Linear(256 * 4 * 4 + 1, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, Y, logE):
        h = self.conv(Y)
        h = h.view(h.size(0), -1)
        h = torch.cat([h, logE], dim=1)
        return self.fc(h)


## Losses

Defines the objective functions. The supervised loss compares log-scaled voxel energies, while the PINN loss adds constraints on total energy, longitudinal shower profile, and radial shower profile.

In [ ]:
# ============================================================
# LOSSES
# ============================================================

def voxel_loss(pred, target):
    return F.mse_loss(torch.log1p(pred), torch.log1p(target))


def energy_loss(pred):
    """
    Y is normalized by E0.
    For contained showers, sum(Y) should be close to 1.
    """
    e = pred.sum(dim=(1, 2, 3))
    return torch.mean((e - 1.0) ** 2)


def longo_loss(pred, E0, longo_targets):
    long_pred = pred.sum(dim=(2, 3))
    long_pred = long_pred / (long_pred.sum(dim=1, keepdim=True) + 1e-12)

    target = get_longo_batch(E0, longo_targets)
    return torch.mean((long_pred - target) ** 2)


def radial_loss(pred):
    rad = radial_project(pred)
    rad = rad / (rad.sum(dim=1, keepdim=True) + 1e-12)

    target = radial_template(rad.shape[1]).to(pred.device)
    return torch.mean((rad - target[None, :]) ** 2)


def pinn_loss(pred, target, E0, longo_targets):
    l_vox = voxel_loss(pred, target)
    l_e = energy_loss(pred)
    l_longo = longo_loss(pred, E0, longo_targets)
    l_rad = radial_loss(pred)

    total = (
        l_vox
        + LAMBDA_E * l_e
        + LAMBDA_LONGO * l_longo
        + LAMBDA_RADIAL * l_rad
    )

    return total, {
        "voxel": l_vox.item(),
        "energy": l_e.item(),
        "longo": l_longo.item(),
        "radial": l_rad.item(),
    }


## Training

Contains the training and evaluation loops for the CNN/PINN models and the adversarial training loop for the GAN. These functions move batches to the selected device, optimize model parameters, and print epoch-level metrics.

In [ ]:
# ============================================================
# TRAINING
# ============================================================

def train_cnn(model, train_loader, valid_loader, epochs, pinn=False, longo_targets=None):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for logE, E0, Y in train_loader:
            logE = logE.to(DEVICE)
            E0 = E0.to(DEVICE)
            Y = Y.to(DEVICE)

            pred = model(logE)

            if pinn:
                loss, logs = pinn_loss(pred, Y, E0, longo_targets)
            else:
                loss = voxel_loss(pred, Y)
                logs = {"voxel": loss.item()}

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_loss += loss.item() * len(Y)

        train_loss /= len(train_loader.dataset)

        valid_loss = evaluate_cnn(model, valid_loader, pinn, longo_targets)

        name = "PINN" if pinn else "CNN"
        print(f"[{name}] epoch {epoch+1:03d} | train={train_loss:.6f} | valid={valid_loss:.6f} | {logs}")


@torch.no_grad()
def evaluate_cnn(model, loader, pinn=False, longo_targets=None):
    model.eval()
    total = 0.0

    for logE, E0, Y in loader:
        logE = logE.to(DEVICE)
        E0 = E0.to(DEVICE)
        Y = Y.to(DEVICE)

        pred = model(logE)

        if pinn:
            loss, _ = pinn_loss(pred, Y, E0, longo_targets)
        else:
            loss = voxel_loss(pred, Y)

        total += loss.item() * len(Y)

    return total / len(loader.dataset)


def train_gan(G, D, train_loader, epochs, noise_dim=64):
    G.to(DEVICE)
    D.to(DEVICE)

    opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

    bce = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        G.train()
        D.train()

        d_loss_sum = 0.0
        g_loss_sum = 0.0

        for logE, E0, real in train_loader:
            logE = logE.to(DEVICE)
            real = real.to(DEVICE)
            B = real.size(0)

            real_labels = torch.ones(B, 1, device=DEVICE)
            fake_labels = torch.zeros(B, 1, device=DEVICE)

            # ---------------------
            # Train discriminator
            # ---------------------
            z = torch.randn(B, noise_dim, device=DEVICE)
            fake = G(logE, z).detach()

            D_real = D(real, logE)
            D_fake = D(fake, logE)

            loss_D = bce(D_real, real_labels) + bce(D_fake, fake_labels)

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # ---------------------
            # Train generator
            # ---------------------
            z = torch.randn(B, noise_dim, device=DEVICE)
            fake = G(logE, z)

            D_fake = D(fake, logE)

            adv_loss = bce(D_fake, real_labels)

            # weak physics stabilizers
            e_loss = energy_loss(fake)
            r_loss = radial_loss(fake)

            loss_G = adv_loss + 0.1 * e_loss + 0.01 * r_loss

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

            d_loss_sum += loss_D.item() * B
            g_loss_sum += loss_G.item() * B

        print(
            f"[GAN] epoch {epoch+1:03d} | "
            f"D={d_loss_sum/len(train_loader.dataset):.6f} | "
            f"G={g_loss_sum/len(train_loader.dataset):.6f}"
        )


## Plots

Contains helper functions for inspecting model quality: average longitudinal profiles, radial profiles, total energy response, and side-by-side event displays with residuals.

In [ ]:
# ============================================================
# PLOTS
# ============================================================

@torch.no_grad()
def get_predictions(model, loader, n_batches=4):
    model.eval()
    ys = []
    ps = []
    e0s = []

    for i, (logE, E0, Y) in enumerate(loader):
        if i >= n_batches:
            break

        logE = logE.to(DEVICE)
        pred = model(logE).cpu()

        ys.append(Y)
        ps.append(pred)
        e0s.append(E0)

    return torch.cat(ys), torch.cat(ps), torch.cat(e0s)


def plot_longitudinal(Y_true, Y_pred, title):
    true = Y_true.sum(dim=(2, 3))
    pred = Y_pred.sum(dim=(2, 3))

    true = true / (true.sum(dim=1, keepdim=True) + 1e-12)
    pred = pred / (pred.sum(dim=1, keepdim=True) + 1e-12)

    plt.figure()
    plt.plot(true.mean(dim=0), label="Geant4")
    plt.plot(pred.mean(dim=0), label="Prediction")
    plt.xlabel("Layer")
    plt.ylabel(r"$E_l / E_{\mathrm{tot}}$")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.show()


def plot_radial_profile(Y_true, Y_pred, title):
    true = radial_project(Y_true)
    pred = radial_project(Y_pred)

    true = true / (true.sum(dim=1, keepdim=True) + 1e-12)
    pred = pred / (pred.sum(dim=1, keepdim=True) + 1e-12)

    plt.figure()
    plt.plot(true.mean(dim=0), label="Geant4")
    plt.plot(pred.mean(dim=0), label="Prediction")
    plt.xlabel("Radial bin")
    plt.ylabel(r"$E(r) / E_{\mathrm{tot}}$")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.show()


def plot_energy_response(Y_true, Y_pred, title):
    e_true = Y_true.sum(dim=(1, 2, 3))
    e_pred = Y_pred.sum(dim=(1, 2, 3))

    ratio = e_pred / (e_true + 1e-12)

    plt.figure()
    plt.hist(ratio.numpy(), bins=50, histtype="step")
    plt.xlabel(r"$E_{\mathrm{pred}} / E_{\mathrm{G4}}$")
    plt.ylabel("Events")
    plt.title(title)
    plt.grid()
    plt.show()


def plot_event_display(Y_true, Y_pred, idx=0, title="Event display"):
    img_true = Y_true[idx].sum(dim=0)
    img_pred = Y_pred[idx].sum(dim=0)
    resid = img_pred - img_true

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    ax[0].imshow(img_true)
    ax[0].set_title("Geant4")

    ax[1].imshow(img_pred)
    ax[1].set_title("Prediction")

    ax[2].imshow(resid)
    ax[2].set_title("Residual")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Main Workflow

Defines the original script workflow as a callable `main()` function. Running `main()` performs data loading, training, evaluation plots, GAN sampling, and model checkpoint saving in one pass.

In [ ]:
def main():
    X_logE, E0, Y = build_arrays(FILES)

    print("Dataset:")
    print("X:", X_logE.shape)
    print("E0:", E0.shape)
    print("Y:", Y.shape)

    longo_targets = fit_longo_targets(Y, E0)

    dataset = CaloDataset(X_logE, E0, Y)

    n_total = len(dataset)
    n_train = int(0.8 * n_total)
    n_valid = int(0.1 * n_total)
    n_test = n_total - n_train - n_valid

    train_set, valid_set, test_set = random_split(
        dataset,
        [n_train, n_valid, n_test],
        generator=torch.Generator().manual_seed(123),
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    # -----------------------------
    # CNN
    # -----------------------------
    cnn = CNNGenerator()
    train_cnn(
        cnn,
        train_loader,
        valid_loader,
        epochs=EPOCHS_CNN,
        pinn=False,
    )

    # -----------------------------
    # PINN
    # -----------------------------
    pinn = CNNGenerator()
    train_cnn(
        pinn,
        train_loader,
        valid_loader,
        epochs=EPOCHS_PINN,
        pinn=True,
        longo_targets=longo_targets,
    )

    # -----------------------------
    # GAN
    # -----------------------------
    G = GANGenerator()
    D = Discriminator()
    train_gan(G, D, train_loader, epochs=EPOCHS_GAN)

    # -----------------------------
    # Plots
    # -----------------------------
    Y_true, Y_cnn, E0_test = get_predictions(cnn, test_loader)
    _, Y_pinn, _ = get_predictions(pinn, test_loader)

    plot_longitudinal(Y_true, Y_cnn, "CNN longitudinal profile")
    plot_longitudinal(Y_true, Y_pinn, "PINN longitudinal profile")

    plot_radial_profile(Y_true, Y_cnn, "CNN radial profile")
    plot_radial_profile(Y_true, Y_pinn, "PINN radial profile")

    plot_energy_response(Y_true, Y_cnn, "CNN energy response")
    plot_energy_response(Y_true, Y_pinn, "PINN energy response")

    plot_event_display(Y_true, Y_cnn, idx=0, title="CNN event display")
    plot_event_display(Y_true, Y_pinn, idx=0, title="PINN event display")

    # GAN sample display
    G.eval()
    with torch.no_grad():
        logE, E0_batch, Y_batch = next(iter(test_loader))
        logE = logE.to(DEVICE)
        z = torch.randn(logE.size(0), 64, device=DEVICE)
        Y_gan = G(logE, z).cpu()

    plot_longitudinal(Y_batch, Y_gan, "GAN longitudinal profile")
    plot_radial_profile(Y_batch, Y_gan, "GAN radial profile")
    plot_energy_response(Y_batch, Y_gan, "GAN energy response")
    plot_event_display(Y_batch, Y_gan, idx=0, title="GAN event display")

    torch.save(cnn.state_dict(), "cnn_calo.pt")
    torch.save(pinn.state_dict(), "pinn_calo.pt")
    torch.save(G.state_dict(), "gan_generator_calo.pt")
    torch.save(D.state_dict(), "gan_discriminator_calo.pt")


## Interactive Workflow

These cells mirror `main()` but let you run training and plotting step by step.

### Prepare Dataset

Loads all configured HDF5 files, converts hits into normalized voxel tensors, fits the Longo targets, creates the dataset, and splits it into train, validation, and test loaders.

In [ ]:
X_logE, E0, Y = build_arrays(FILES)

print("Dataset:")
print("X:", X_logE.shape)
print("E0:", E0.shape)
print("Y:", Y.shape)

longo_targets = fit_longo_targets(Y, E0)

dataset = CaloDataset(X_logE, E0, Y)

n_total = len(dataset)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)
n_test = n_total - n_train - n_valid

train_set, valid_set, test_set = random_split(
    dataset,
    [n_train, n_valid, n_test],
    generator=torch.Generator().manual_seed(123),
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)


### Train CNN

Initializes and trains the supervised CNN surrogate using only the voxel reconstruction loss.

In [ ]:
cnn = CNNGenerator()
train_cnn(
    cnn,
    train_loader,
    valid_loader,
    epochs=EPOCHS_CNN,
    pinn=False,
)


### Train PINN

Initializes and trains the physics-informed CNN surrogate using the voxel loss plus energy, longitudinal-profile, and radial-profile penalties.

In [ ]:
pinn = CNNGenerator()
train_cnn(
    pinn,
    train_loader,
    valid_loader,
    epochs=EPOCHS_PINN,
    pinn=True,
    longo_targets=longo_targets,
)


### Train GAN

Initializes the conditional GAN generator and discriminator, then trains them adversarially with weak physics stabilizers on generated showers.

In [ ]:
G = GANGenerator()
D = Discriminator()
train_gan(G, D, train_loader, epochs=EPOCHS_GAN)


### Evaluate and Plot

Generates predictions from the trained models, plots comparison diagnostics, samples the GAN on test energies, and saves all trained model weights to `.pt` checkpoint files.

In [ ]:
Y_true, Y_cnn, E0_test = get_predictions(cnn, test_loader)
_, Y_pinn, _ = get_predictions(pinn, test_loader)

plot_longitudinal(Y_true, Y_cnn, "CNN longitudinal profile")
plot_longitudinal(Y_true, Y_pinn, "PINN longitudinal profile")

plot_radial_profile(Y_true, Y_cnn, "CNN radial profile")
plot_radial_profile(Y_true, Y_pinn, "PINN radial profile")

plot_energy_response(Y_true, Y_cnn, "CNN energy response")
plot_energy_response(Y_true, Y_pinn, "PINN energy response")

plot_event_display(Y_true, Y_cnn, idx=0, title="CNN event display")
plot_event_display(Y_true, Y_pinn, idx=0, title="PINN event display")

# GAN sample display
G.eval()
with torch.no_grad():
    logE, E0_batch, Y_batch = next(iter(test_loader))
    logE = logE.to(DEVICE)
    z = torch.randn(logE.size(0), 64, device=DEVICE)
    Y_gan = G(logE, z).cpu()

plot_longitudinal(Y_batch, Y_gan, "GAN longitudinal profile")
plot_radial_profile(Y_batch, Y_gan, "GAN radial profile")
plot_energy_response(Y_batch, Y_gan, "GAN energy response")
plot_event_display(Y_batch, Y_gan, idx=0, title="GAN event display")

torch.save(cnn.state_dict(), "cnn_calo.pt")
torch.save(pinn.state_dict(), "pinn_calo.pt")
torch.save(G.state_dict(), "gan_generator_calo.pt")
torch.save(D.state_dict(), "gan_discriminator_calo.pt")
